In [ ]:
def main():
    # experiment_runner
    from autora.experiment_runner.synthetic.psychophysics.weber_fechner_law import weber_fechner_law
    from autora.experiment_runner.synthetic.psychophysics.stevens_power_law import stevens_power_law
    from autora.experiment_runner.synthetic.economics.expected_value_theory import expected_value_theory
    from autora.state import StandardState, on_state


    # data handling
    from sklearn.model_selection import train_test_split



    def benchmark(experiment_runner, theorist):
        # DEMONSTRATION: FITTING Part 1
        @on_state(output=['models'])
        def theorist_on_state(experiment_data, variables):
            benchmark_theorist = theorist.fit(experiment_data, variables)
            return benchmark_theorist

        conditions = experiment_runner.domain()
        # generate all corresponding observations
        experiment_data = experiment_runner.run(conditions, added_noise=0.001)
        state = StandardState(variables=experiment_runner.variables, conditions=conditions, experiment_data=experiment_data, models=[])
        #DEMONSTRATION: FITTING Part 2 (theorist_on_state function is called here)
        state = theorist_on_state(state)
        benchmark_theorist = state.models[-1]
        # evaluate theorist performance
        independent_var_names = [var.name for var in experiment_runner.variables.independent_variables]
        dependent_var_names = [var.name for var in experiment_runner.variables.dependent_variables]
        X = state.experiment_data[independent_var_names]
        y = state.experiment_data[dependent_var_names]
        #DEMONSTRATION: PREDICTION
        y_pred = benchmark_theorist.predict(X)
        error = ((y - y_pred) ** 2).mean().values[0]
        print(f"Theorist: {theorist.__class__.__name__}, Validation MSE: {error}")
        #DEMONSTRATION: Obtaining the equation
        winning_equation = benchmark_theorist.repr()
        print(f"Winning Equation: {winning_equation}")



    #%%
    # from autora.theorist.autora_theorist_polynomial import ExampleRegressor

    my_theorist = ParallelizedAkaikeBMSRegressorPool(epochs=1500, pool_size=24)
    #my_theorist = BMSRegressor(epochs=10)  # simpler baseline
    #my_theorist = dummy_theorist # remove this once you implemented your own theorist
    print("Benchmarking Stevens' Power Law")
    benchmark(experiment_runner=stevens_power_law(), theorist=my_theorist)
    print("Benchmarking Weber-Fechner Law")
    benchmark(experiment_runner=weber_fechner_law(), theorist=my_theorist)
    print("Benchmarking Expected Value Theory")
    benchmark(experiment_runner=expected_value_theory(), theorist=my_theorist)